# Appendix C: Numerical Linear Algebra Background

## Overview
Efficiently solving convex optimization problems, especially via interior-point methods, relies heavily on numerical linear algebra. The bottleneck in these algorithms is usually solving a large system of linear equations (the KKT system). This appendix reviews the standard matrix factorizations used to solve these systems efficiently and stably.

### Key Concepts

1. **Solving Linear Equations:** To solve $Ax = b$, we rarely compute $A^{-1}$ explicitly. Instead, we factor $A$ into a product of simpler matrices (e.g., lower and upper triangular) and solve the system using forward and backward substitution.

2. **Cholesky Factorization:** If $A$ is symmetric positive definite ($A \succ 0$), it can be factored as:
   
$$
A = L L^T
$$
   
   where $L$ is a lower triangular matrix. This is the gold standard for solving systems involving the Hessian in unconstrained minimization or the normal equations. It is extremely fast and numerically stable.

3. **LU and $LDL^T$ Factorizations:** 
   - **LU Factorization:** Used for general square, non-singular matrices ($A = PLU$, where $P$ is a permutation matrix).
   - **$LDL^T$ Factorization:** Used for symmetric indefinite matrices (like the KKT matrix in equality constrained optimization). $A = P L D L^T P^T$, where $D$ is block diagonal.

4. **Singular Value Decomposition (SVD):** Any matrix $A \in \mathbb{R}^{m \times n}$ can be factored as:
   
$$
A = U \Sigma V^T
$$
   
   where $U$ and $V$ are orthogonal matrices, and $\Sigma$ is a diagonal matrix of singular values. SVD reveals the fundamental geometric transformation applied by a matrix: a rotation ($V^T$), followed by independent scaling along the axes ($\Sigma$), followed by another rotation ($U$).

5. **Block Elimination and Schur Complement:** When solving systems with block structure (like the KKT system), we can use block elimination. The key algebraic object that arises is the Schur complement $S = C - B^T A^{-1} B$, which allows us to solve a smaller system first.

## Code Example
See `matrix_factorizations.py` for a geometric visualization of the **Singular Value Decomposition (SVD)**. We apply a random $2 \times 2$ matrix to a unit circle of points and decompose the transformation step-by-step into its $V^T$ (rotation), $\Sigma$ (scaling), and $U$ (final rotation) components.

![SVD Transformation](svd_transformation.png)


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def plot_circle(ax, points, color='blue', title=''):
    ax.plot(points[0, :], points[1, :], color=color, lw=2)
    
    # Plot standard basis vectors to see how they rotate
    ax.arrow(0, 0, points[0, 0], points[1, 0], head_width=0.1, color='red', lw=2)
    ax.arrow(0, 0, points[0, 25], points[1, 25], head_width=0.1, color='green', lw=2)
    
    ax.set_aspect('equal')
    ax.set_xlim([-4, 4])
    ax.set_ylim([-4, 4])
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_title(title)
    ax.axhline(0, color='black', lw=0.5)
    ax.axvline(0, color='black', lw=0.5)

def demonstrate_svd():
    """
    Geometrically demonstrates the SVD of a 2x2 matrix: A = U * Sigma * V^T
    """
    # Create a unit circle
    theta = np.linspace(0, 2*np.pi, 100)
    circle = np.vstack([np.cos(theta), np.sin(theta)])
    
    # Define a 2x2 matrix A
    A = np.array([[1.5, 0.5], 
                  [-1.0, 2.0]])
    
    # Compute SVD
    U, S, VT = np.linalg.svd(A)
    Sigma = np.diag(S)
    
    # Step-by-step transformations
    # 1. V^T applied to the circle (Rotation)
    circle_vt = VT @ circle
    
    # 2. Sigma applied to the result (Scaling)
    circle_sigma = Sigma @ circle_vt
    
    # 3. U applied to the result (Rotation) -> This equals A * circle
    circle_u = U @ circle_sigma
    
    # Plotting
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    plot_circle(axes[0], circle, color='blue', title='Original Unit Circle ($x$)')
    plot_circle(axes[1], circle_vt, color='purple', title='1. Rotation ($V^T x$)')
    plot_circle(axes[2], circle_sigma, color='orange', title='2. Scaling ($\Sigma V^T x$)')
    plot_circle(axes[3], circle_u, color='red', title='3. Rotation ($U \Sigma V^T x = Ax$)')
    
    plt.tight_layout()
    plt.savefig("svd_transformation.png", dpi=150)
    print("Plot saved as svd_transformation.png")

if __name__ == "__main__":
    demonstrate_svd()
